In [24]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from tqdm import tqdm
import warnings
from scipy.optimize import minimize
warnings.filterwarnings('ignore')

np.random.seed(42)

In [25]:
train_df = pd.read_csv('data/MR_number_train_0w-5w.csv.zip', index_col=0)
test_df = pd.read_csv('data/MR_number_test_5w-6w.csv.zip', index_col=0)

In [26]:
train_data = train_df.values.astype(np.float32)
test_data = test_df.values.astype(np.float32)

print(f"Train: {train_data.shape}, Test: {test_data.shape}")
print(f"Train zeros: {(train_data == 0).sum() / train_data.size * 100:.1f}%")

Train: (840, 2880), Test: (168, 2880)
Train zeros: 20.2%


# Улучшенная сезонная модель c обучаемыми весами

In [27]:
class LearnableSeasonal:
    def __init__(self):
        self.daily_weights = None
        self.weekly_weights = None

    def fit(self, train_data, val_data):
        train_arr = train_data.values if hasattr(train_data, 'values') else train_data
        val_arr = val_data.values if hasattr(val_data, 'values') else val_data

        def loss(weights):
            daily_w, weekly_w = weights[0], weights[1]
            preds = []
            for t in range(len(val_arr)):
                daily_idx = len(train_arr) - 24 + (t % 24)
                weekly_idx = len(train_arr) - 168 + (t % 168)
                daily_val = train_arr[daily_idx] if daily_idx >= 0 else train_arr[-1]
                weekly_val = train_arr[weekly_idx] if weekly_idx >= 0 else train_arr[-1]
                pred = daily_w * daily_val + weekly_w * weekly_val
                preds.append(pred)
            preds = np.array(preds)
            return mean_absolute_error(val_arr.flatten(), preds.flatten())

        result = minimize(loss, [0.6, 0.4], bounds=[(0,1), (0,1)], method='L-BFGS-B')
        self.daily_weight, self.weekly_weight = result.x
        print(f"Optimized weights: daily={self.daily_weight:.3f}, weekly={self.weekly_weight:.3f}")
        return self

    def predict(self, train_data, steps=168):
        train_arr = train_data.values if hasattr(train_data, 'values') else train_data
        preds = []
        for t in range(steps):
            daily_idx = len(train_arr) - 24 + (t % 24)
            weekly_idx = len(train_arr) - 168 + (t % 168)
            daily_val = train_arr[daily_idx] if daily_idx >= 0 else train_arr[-1]
            weekly_val = train_arr[weekly_idx] if weekly_idx >= 0 else train_arr[-1]
            pred = self.daily_weight * daily_val + self.weekly_weight * weekly_val
            preds.append(pred)
        return np.array(preds)

In [28]:
val_data = train_data[-168:]  # последние 168 часов train
train_for_seasonal = train_data[:-168]

seasonal_model = LearnableSeasonal()
seasonal_model.fit(train_df.iloc[:-168], val_data)

Optimized weights: daily=0.444, weekly=0.506


## Предсказания для теста (на основе train)

In [29]:
seasonal_pred = seasonal_model.predict(train_df)

seasonal_pred имеет длину 168, но ожидает форму (168, 2880). Исправляем размерность

In [30]:
if seasonal_pred.ndim == 1:
    seasonal_pred = seasonal_pred.reshape(-1, 1)
    # Повторяем для всех лучей (базовое предсказание одинаково для всех)
    seasonal_pred = np.tile(seasonal_pred, (1, train_data.shape[1]))

# Линейная регнессия

In [31]:
# Prepare features: use last 24h + rolling stats
def prepare_lr_features(data, window=24):
    X, y = [], []
    for t in range(window, len(data)):
        # Last 24h
        features = data[t-window:t].flatten()
        # Add rolling mean and std
        features = np.append(features, data[t-window:t].mean(axis=0))
        features = np.append(features, data[t-window:t].std(axis=0))
        X.append(features)
        y.append(data[t])
    return np.array(X), np.array(y)

#Обучаем ТОЛЬКО на train_data
X_train, y_train = prepare_lr_features(train_data, window=24)
print(f"LR train shape: {X_train.shape}")

lr_model = Ridge(alpha=0.1, random_state=42)
lr_model.fit(X_train, y_train)

#Предсказываем ТОЛЬКО на основе train_data
lr_pred = np.zeros((168, train_data.shape[1]))
last_window = train_data[-24:].copy()

for t in range(168):
    features = last_window.flatten()
    features = np.append(features, last_window.mean(axis=0))
    features = np.append(features, last_window.std(axis=0))
    pred = lr_model.predict(features.reshape(1, -1))[0]
    lr_pred[t] = pred
    last_window = np.vstack([last_window[1:], pred])

lr_pred = np.maximum(lr_pred, 0)

LR train shape: (816, 74880)


# xgboost

In [32]:
xgb_pred = np.zeros((168, train_data.shape[1]))

num_beams = train_data.shape[1]
beam_activity = train_data.mean(axis=0)
most_active = np.argsort(beam_activity)[-num_beams:]

for beam_idx in tqdm(most_active, desc="XGBoost"):
    beam_data = train_data[:, beam_idx]
    # Use log transform for better handling of zeros
    beam_log = np.log1p(beam_data)
    # Prepare features
    X_beam, y_beam = [], []
    for t in range(168, len(beam_log)):
        features = beam_log[t-168:t].tolist()
        features.append(beam_log[t-24:t].mean())
        features.append(beam_log[t-168:t].mean())
        features.append(beam_log[t-24:t].std())
        X_beam.append(features)
        y_beam.append(beam_log[t])
    X_beam = np.array(X_beam)
    y_beam = np.array(y_beam)
    # Train XGBoost
    model = XGBRegressor(
        n_estimators=50,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42,
        n_jobs=1, # because of issue in mac Mx CPU family
        verbosity=0
    )
    model.fit(X_beam, y_beam)

    # Predict recursively starting from train_data
    last_168 = beam_log[-168:]
    preds_log = []
    current = last_168.copy()

    for _ in range(168):
        features = current.tolist()
        features.append(current[-24:].mean())
        features.append(current.mean())
        features.append(current[-24:].std())
        pred_log = model.predict(np.array(features).reshape(1, -1))[0]
        preds_log.append(pred_log)
        current = np.roll(current, -1)
        current[-1] = pred_log

    xgb_pred[:, beam_idx] = np.expm1(preds_log)

XGBoost: 100%|██████████| 2880/2880 [18:46<00:00,  2.56it/s]


# Ансамблевая модель

Валидация на последних 168 часах train_data

In [33]:
val_actual = train_data[-168:]
val_pred_seasonal = seasonal_model.predict(train_df.iloc[:-168])
if val_pred_seasonal.ndim == 1:
    val_pred_seasonal = val_pred_seasonal.reshape(-1, 1)
    val_pred_seasonal = np.tile(val_pred_seasonal, (1, train_data.shape[1]))

In [34]:
val_pred_lr = lr_pred  # lr_pred уже предсказан на основе train_data
val_pred_xgb = xgb_pred  # xgb_pred уже предсказан на основе train_data

In [35]:
def ensemble_loss(weights):
    w_s, w_l, w_x = weights
    w_sum = w_s + w_l + w_x
    if w_sum == 0:
        return 1e6
    w_s, w_l, w_x = w_s/w_sum, w_l/w_sum, w_x/w_sum
    pred = (w_s * val_pred_seasonal +
            w_l * val_pred_lr +
            w_x * val_pred_xgb)
    return mean_absolute_error(val_actual.flatten(), pred.flatten())

result = minimize(ensemble_loss, [0.2, 0.3, 0.5],
                  bounds=[(0,1), (0,1), (0,1)],
                  method='L-BFGS-B')

w_s, w_l, w_x = result.x
w_sum = w_s + w_l + w_x
w_s, w_l, w_x = w_s/w_sum, w_l/w_sum, w_x/w_sum
print(f"Optimal weights:")
print(f"Seasonal: {w_s:.3f}")
print(f"Linear Regression: {w_l:.3f}")
print(f"XGBoost: {w_x:.3f}")

Optimal weights:
Seasonal: 0.000
Linear Regression: 0.544
XGBoost: 0.456


## Итоговый ансамбль

In [36]:
ensemble_pred = (w_s * seasonal_pred +
                 w_l * lr_pred +
                 w_x * xgb_pred)

# Apply daily pattern smoothing (based ONLY on train_data)
daily_pattern = np.zeros((24, train_data.shape[1]))
for hour in range(24):
    hour_indices = np.arange(hour, len(train_data), 24)
    if len(hour_indices) > 0:
        daily_pattern[hour] = train_data[hour_indices].mean(axis=0)

# Smooth predictions using daily pattern
for t in range(168):
    hour = t % 24
    ensemble_pred[t] = 0.85 * ensemble_pred[t] + 0.15 * daily_pattern[hour]

ensemble_pred = np.maximum(ensemble_pred, 0)

используем test_df для вычисления метрик

### Регрессионные метрики

In [37]:
ensemble_mae = mean_absolute_error(test_data.flatten(), ensemble_pred.flatten())
ensemble_rmse = np.sqrt(np.mean((test_data.flatten() - ensemble_pred.flatten())**2))
print(f"Ensemble MAE:  {ensemble_mae:.6f}")
print(f"Ensemble RMSE: {ensemble_rmse:.6f}")

Ensemble MAE:  0.215746
Ensemble RMSE: 0.377551


### Бинарная классификация

In [38]:
test_actual = test_df.values.astype(np.float32)

THRESHOLD = 0.5

y_true_binary = (test_actual > THRESHOLD).astype(int)
y_pred_binary = (ensemble_pred > THRESHOLD).astype(int)

precision = precision_score(y_true_binary.flatten(), y_pred_binary.flatten())
recall = recall_score(y_true_binary.flatten(), y_pred_binary.flatten())
f1 = f1_score(y_true_binary.flatten(), y_pred_binary.flatten())
accuracy = accuracy_score(y_true_binary.flatten(), y_pred_binary.flatten())

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

Precision: 0.8519
Recall: 0.8277
F1-Score: 0.8396
Accuracy: 0.8991


### Confusion Matrix

In [39]:
cm = confusion_matrix(y_true_binary.flatten(), y_pred_binary.flatten())
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives:  {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives:  {tp:,}")

specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
print(f"Specificity (TNR): {specificity:.4f}")

True Negatives:  307,288
False Positives: 22,211
False Negatives: 26,599
True Positives:  127,742
Specificity (TNR): 0.9326


### Precision-recall curve

In [40]:
y_true_flat = test_actual.flatten()
y_pred_flat = ensemble_pred.flatten()

precision_curve, recall_curve, thresholds = precision_recall_curve(
    y_true_flat > THRESHOLD,
    y_pred_flat
)

ap_score = average_precision_score(y_true_flat > THRESHOLD, y_pred_flat)
print(f"Average Precision (AP): {ap_score:.4f}")

f1_scores = 2 * (precision_curve[:-1] * recall_curve[:-1]) / (precision_curve[:-1] + recall_curve[:-1] + 1e-8)
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_idx] if len(thresholds) > 0 else THRESHOLD
best_f1 = f1_scores[best_threshold_idx]

print(f"Best F1-Score: {best_f1:.4f} at threshold: {best_threshold:.4f}")

Average Precision (AP): 0.9260
Best F1-Score: 0.8404 at threshold: 0.4842


### Activity patterns

In [41]:
actual_active_rate = (test_actual > THRESHOLD).mean()
pred_active_rate = (ensemble_pred > THRESHOLD).mean()

print(f"Actual active rate: {actual_active_rate:.2%}")
print(f"Predicted active rate: {pred_active_rate:.2%}")
print(f"Bias: {pred_active_rate - actual_active_rate:+.2%}")

Actual active rate: 31.90%
Predicted active rate: 30.99%
Bias: -0.91%


### Activity patterns по часам

In [42]:
hourly_actual_active = []
hourly_pred_active = []

for hour in range(24):
    hour_indices = range(hour, len(test_actual), 24)
    hourly_actual_active.append((test_actual[hour_indices] > THRESHOLD).mean())
    hourly_pred_active.append((ensemble_pred[hour_indices] > THRESHOLD).mean())

print("Activity by hour of day:")
for hour in [0, 6, 12, 18, 23]:
    print(f"Hour {hour:2d}: Actual: {hourly_actual_active[hour]:.2%}, Predicted: {hourly_pred_active[hour]:.2%}")

Activity by hour of day:
Hour  0: Actual: 23.58%, Predicted: 22.40%
Hour  6: Actual: 24.38%, Predicted: 22.98%
Hour 12: Actual: 38.36%, Predicted: 38.15%
Hour 18: Actual: 39.42%, Predicted: 38.45%
Hour 23: Actual: 27.67%, Predicted: 26.38%


### Разные пороги

In [43]:
thresholds_to_test = [0, 0.5, 1.0, 2.0, 5.0]

results_summary = []
for thr in thresholds_to_test:
    y_true_thr = (test_actual > thr).astype(int)
    y_pred_thr = (ensemble_pred > thr).astype(int)

    p = precision_score(y_true_thr.flatten(), y_pred_thr.flatten(), zero_division=0)
    r = recall_score(y_true_thr.flatten(), y_pred_thr.flatten(), zero_division=0)
    f = f1_score(y_true_thr.flatten(), y_pred_thr.flatten(), zero_division=0)

    results_summary.append({
        'Threshold': thr,
        'Precision': p,
        'Recall': r,
        'F1-Score': f,
        'Actual_Active_Rate': y_true_thr.mean(),
        'Pred_Active_Rate': y_pred_thr.mean()
    })

results_df = pd.DataFrame(results_summary)
results_df

,Threshold,Precision,Recall,F1-Score,Actual_Active_Rate,Pred_Active_Rate
0,0.0,0.800887,1.000000,0.889436,0.800887,1.000000
1,0.5,0.851880,0.827661,0.839596,0.318992,0.309923
2,1.0,0.839172,0.814638,0.826723,0.180130,0.174864
3,2.0,0.840889,0.782480,0.810634,0.075548,0.070300
4,5.0,0.814064,0.672876,0.736767,0.011095,0.009170


### Detailed classification report

In [44]:
print(classification_report(
    y_true_binary.flatten(),
    y_pred_binary.flatten(),
    target_names=['Inactive (0 users)', 'Active (>0 users)']
))

                    precision    recall  f1-score   support

Inactive (0 users)       0.92      0.93      0.93    329499
 Active (>0 users)       0.85      0.83      0.84    154341

          accuracy                           0.90    483840
         macro avg       0.89      0.88      0.88    483840
      weighted avg       0.90      0.90      0.90    483840



### Бизнесовые метрики

In [45]:
wasted_resources = fp / (tp + fp) if (tp + fp) > 0 else 0
missed_opportunities = fn / (tp + fn) if (tp + fn) > 0 else 0

print(f"Resource Waste Rate: {wasted_resources:.2%}")
print(f"Missed Opportunity Rate: {missed_opportunities:.2%}")

cost_fp = 1.0
cost_fn = 5.0
total_cost = (fp * cost_fp + fn * cost_fn) / len(y_true_flat)
print(f"Weighted Error Cost: {total_cost:.4f} per prediction")

Resource Waste Rate: 14.81%
Missed Opportunity Rate: 17.23%
Weighted Error Cost: 0.3208 per prediction


# Выводы

## Резюме

Ансамблевая модель демонстрирует высокую предсказательную способность как в задачах регрессии, так и в бинарной классификации. При MAE = 0.216 и F1-Score = 0.840 модель успешно балансирует между точностью предсказания точного числа пользователей и надежным определением активности лучей.

## Регрессионные метрики

Модель достигает средней абсолютной ошибки (MAE) 0.216 и среднеквадратичной ошибки (RMSE) 0.378 на тестовых данных. Это означает, что предсказания отклоняются от фактического числа пользователей в среднем на 0.22 пользователя. Значение RMSE умеренно превышает MAE, что указывает на наличие небольшого количества более крупных ошибок, но в целом точность прогнозирования является высокой.

## Метрики классификации при оптимальном пороге (0.5)

- Precision: 85.19% — когда модель предсказывает луч как активный, она права в 5 из 6 случаев
- Recall: 82.77% — модель обнаруживает почти 83% всех реально активных лучей
- F1: 84.0% — отличный баланс между точностью и полнотой
- Accuracy: 89.91% — почти 90% всех предсказаний верны

## Суточные паттерны активности

Анализ по часам суток подтверждает наличие выраженной сезонности:

| Час | Фактическая активность | Предсказанная активность | Расхождение |
|-----|------------------------|--------------------------|-------------|
| 0:00 | 23.58% | 22.40% | -1.18% |
| 6:00 | 24.38% | 22.98% | -1.40% |
| 12:00 | 38.36% | 38.15% | -0.21% |
| 18:00 | 39.42% | 38.45% | -0.97% |
| 23:00 | 27.67% | 26.38% | -1.29% |

Модель хорошо улавливает суточную динамику: пик активности приходится на дневные часы (12:00-18:00) с уровнем около 38-39%, а минимум — на ночные часы (23-24%). Расхождение между фактическими и предсказанными значениями не превышает 1.5% ни в одном из часовых интервалов.

## Анализ при различных порогах активации

| Порог | Точность | Полнота | F1-Score | Фактическая активность | Предсказанная активность |
|-------|----------|---------|----------|------------------------|--------------------------|
| 0 | 80.09% | 100.00% | 88.94% | 80.09% | 100.00% |
| 0.5 | 85.19% | 82.77% | 83.96% | 31.90% | 30.99% |
| 1.0 | 83.92% | 81.46% | 82.67% | 18.01% | 17.49% |
| 2.0 | 84.09% | 78.25% | 81.06% | 7.55% | 7.03% |
| 5.0 | 81.41% | 67.29% | 73.68% | 1.11% | 0.92% |

Модель сохраняет F1-Score выше 0.81 для всех порогов от 0.5 до 2.0, что свидетельствует о стабильности и робастности. При пороге 0.5 достигается наилучший баланс между точностью и полнотой для практического использования.

# Итого

1. Высокая точность обнаружения неактивных лучей (специфичность 93.3%)
2. Стабильность при изменении порога принятия решений
3. Корректное воспроизведение суточных паттернов активности
4. Качество ранжирования (AP = 92.6%) — модель хорошо отделяет активные лучи от неактивных